<a href="https://colab.research.google.com/github/oxigii/BigData26/blob/main/task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")

print(df.head())
print(df.shape)


In [ ]:
missing = df.isnull().sum()

print(missing)
(df.isnull().sum()/len(df))*100

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
sns.heatmap(df.isnull(), cbar=False)
plt.show()

###이상치 탐색
Age

In [ ]:
sns.boxplot(x=df['Age'])
plt.show()

Fare

In [ ]:
sns.boxplot(x=df['Fare'])
plt.show()

###변수 분포 시각화

In [ ]:
sns.histplot(df['Age'], kde=True)
plt.show()

sns.histplot(df['Fare'], kde=True)
plt.show()

###상관관계 분석

In [ ]:
numeric_df = df.select_dtypes(include=['int64','float64'])

corr = numeric_df.corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr,
            annot=True,
            cmap='coolwarm')
plt.show()

###타겟 변수 분포

In [ ]:
sns.countplot(x='Survived', data=df)
plt.show()

df['Survived'].value_counts(normalize=True)

###3 결측치 처리 비교
Mean/Median

In [ ]:
print("처리 전 결측치 수")
print(df['Age'].isnull().sum())

In [ ]:
from sklearn.impute import SimpleImputer

mean_df = df.copy()

mean_imputer = SimpleImputer(strategy='mean')

mean_df['Age'] = mean_imputer.fit_transform(
    mean_df[['Age']]
)

In [ ]:
print("처리 후 결측치 수")
print(mean_df['Age'].isnull().sum())

In [ ]:
print("중앙값")
print(df['Age'].median())

In [ ]:
median_df = df.copy()

median_imputer = SimpleImputer(strategy='median')

median_df['Age'] = median_imputer.fit_transform(
    median_df[['Age']]
)

In [ ]:
print("처리 후 결측치")
print(median_df['Age'].isnull().sum())

Most Freauent

In [ ]:
print("최빈값")
print(df['Age'].mode()[0])

In [ ]:
most_df = df.copy()

most_imputer = SimpleImputer(strategy='most_frequent')

most_df['Age'] = most_imputer.fit_transform(
    most_df[['Age']]
)

In [ ]:
print("처리 후 결측치")
print(most_df['Age'].isnull().sum())

###3-2 범주형 인코딩 비교

Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_df = mean_df.copy()

le = LabelEncoder()

label_df['Sex'] = le.fit_transform(label_df['Sex'])

label_df['Embarked'] = le.fit_transform(
    label_df['Embarked']
)

print(label_df[['Sex','Embarked']].head())

One-hot Encoding

In [ ]:
onehot_df = mean_df.copy()

onehot_df = pd.get_dummies(
    onehot_df,
    columns=['Sex','Embarked'],
    drop_first=True
)

print(onehot_df.columns)

###3-3 스케일링 비교

StandardScaler

In [ ]:
print(df[['Age','Fare']].describe())

In [ ]:
from sklearn.preprocessing import StandardScaler

standard_df = label_df.copy()

scaler = StandardScaler()

cols = ['Age','Fare']

standard_df[cols] = scaler.fit_transform(
    standard_df[cols]
)

In [ ]:
print(standard_df[['Age','Fare']].describe())

MinMaxScaler

In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax_df = label_df.copy()

scaler = MinMaxScaler()

minmax_df[cols] = scaler.fit_transform(
    minmax_df[cols]
)

In [ ]:
print(minmax_df[['Age','Fare']].describe())

###3-4 파생 변수 생성

In [ ]:
feature_df = df.copy()

feature_df['FamilySize'] = (
    feature_df['SibSp']
    + feature_df['Parch']
    + 1
)

feature_df['IsAlone'] = (
    feature_df['FamilySize'] == 1
).astype(int)

feature_df['FarePerPerson'] = (
    feature_df['Fare']
    / feature_df['FamilySize']
)

feature_df['AgeGroup'] = pd.cut(
    feature_df['Age'],
    bins=[0,12,19,39,59,100],
    labels=[
        'Child',
        'Teen',
        'Adult',
        'Middle',
        'Senior'
    ]
)

In [ ]:
print(feature_df[
    ['FamilySize',
     'IsAlone',
     'FarePerPerson']
].head())

In [ ]:
print(
    feature_df['AgeGroup']
    .value_counts()
)

In [ ]:
print(df[['Age','Fare']].describe())
print(standard_df[['Age','Fare']].describe())

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")

# 불필요 컬럼 제거
df = df.drop(
    columns=[
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin'
    ]
)

print(df.head())

##Exp-1 : Mean + OneHot + Standard

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

exp1 = df.copy()

# 결측치 처리
exp1['Age'] = SimpleImputer(
    strategy='mean'
).fit_transform(exp1[['Age']])

exp1['Embarked'] = SimpleImputer(
    strategy='most_frequent'
).fit_transform(exp1[['Embarked']]).ravel()

# 파생변수 생성
exp1['FamilySize'] = (
    exp1['SibSp']
    + exp1['Parch']
    + 1
)

exp1['FarePerPerson'] = (
    exp1['Fare']
    / exp1['FamilySize']
)

# One-Hot Encoding
exp1 = pd.get_dummies(
    exp1,
    columns=['Sex','Embarked'],
    drop_first=True
)

# StandardScaler
num_cols = [
    'Age',
    'Fare',
    'FamilySize',
    'FarePerPerson'
]

scaler = StandardScaler()

exp1[num_cols] = scaler.fit_transform(
    exp1[num_cols]
)

print(exp1.head())
print(exp1.shape)

##Exp-2 : Median + Label + MinMax

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

exp2 = df.copy()

# 결측치
exp2['Age'] = SimpleImputer(
    strategy='median'
).fit_transform(exp2[['Age']])

exp2['Embarked'] = SimpleImputer(
    strategy='most_frequent'
).fit_transform(exp2[['Embarked']]).ravel()

# 파생변수
exp2['FamilySize'] = (
    exp2['SibSp']
    + exp2['Parch']
    + 1
)

exp2['FarePerPerson'] = (
    exp2['Fare']
    / exp2['FamilySize']
)

# Label Encoding
le = LabelEncoder()

exp2['Sex'] = le.fit_transform(
    exp2['Sex']
)

exp2['Embarked'] = le.fit_transform(
    exp2['Embarked']
)

# MinMax Scaling
num_cols = [
    'Age',
    'Fare',
    'FamilySize',
    'FarePerPerson'
]

scaler = MinMaxScaler()

exp2[num_cols] = scaler.fit_transform(
    exp2[num_cols]
)

print(exp2.head())
print(exp2.shape)

##Exp-3 : Most Frequent + OneHot + Robust

In [ ]:
from sklearn.preprocessing import RobustScaler

exp3 = df.copy()

# 결측치
exp3['Age'] = SimpleImputer(
    strategy='most_frequent'
).fit_transform(exp3[['Age']])

exp3['Embarked'] = SimpleImputer(
    strategy='most_frequent'
).fit_transform(exp3[['Embarked']]).ravel()

# 파생변수
exp3['FamilySize'] = (
    exp3['SibSp']
    + exp3['Parch']
    + 1
)

exp3['FarePerPerson'] = (
    exp3['Fare']
    / exp3['FamilySize']
)

# One-Hot Encoding
exp3 = pd.get_dummies(
    exp3,
    columns=['Sex','Embarked'],
    drop_first=True
)

# Robust Scaling
num_cols = [
    'Age',
    'Fare',
    'FamilySize',
    'FarePerPerson'
]

scaler = RobustScaler()

exp3[num_cols] = scaler.fit_transform(
    exp3[num_cols]
)

print(exp3.head())
print(exp3.shape)

In [ ]:
print(exp1.isnull().sum().sum())
print(exp2.isnull().sum().sum())
print(exp3.isnull().sum().sum())

##4. 변수 선택
###Random Forest Feature Importace

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

X = exp1.drop('Survived', axis=1)
y = exp1['Survived']

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X, y)

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
})

importance = importance.sort_values(
    'Importance',
    ascending=False
)

print(importance)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))

sns.barplot(
    data=importance.head(10),
    x='Importance',
    y='Feature'
)

plt.title("Feature Importance")
plt.show()

In [ ]:
top_features = (
    importance
    .head(5)['Feature']
    .tolist()
)

print(top_features)

In [ ]:
X_full = exp1.drop('Survived',axis=1)
X_full

In [ ]:
X_selected = exp1[
    top_features
]

X_selected

##5. 모델 학습 및 평가
Logistic Regression

In [ ]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

df = pd.read_csv("train.csv")

df = df.drop(
    columns=[
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin'
    ]
)


def add_features(data):
    data = data.copy()

    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1

    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)

    data['FarePerPerson'] = data['Fare'] / data['FamilySize']

    return data

def make_exp1(df):
    exp = df.copy()

    exp['Age'] = SimpleImputer(strategy='mean').fit_transform(exp[['Age']])
    exp['Embarked'] = SimpleImputer(strategy='most_frequent').fit_transform(exp[['Embarked']]).ravel()

    exp = add_features(exp)

    exp = pd.get_dummies(
        exp,
        columns=['Sex', 'Embarked'],
        drop_first=True
    )

    num_cols = ['Age', 'Fare', 'FamilySize', 'FarePerPerson']

    scaler = StandardScaler()
    exp[num_cols] = scaler.fit_transform(exp[num_cols])

    return exp


def make_exp2(df):
    exp = df.copy()

    exp['Age'] = SimpleImputer(strategy='median').fit_transform(exp[['Age']])
    exp['Embarked'] = SimpleImputer(strategy='most_frequent').fit_transform(exp[['Embarked']]).ravel()

    exp = add_features(exp)

    le_sex = LabelEncoder()
    le_embarked = LabelEncoder()

    exp['Sex'] = le_sex.fit_transform(exp['Sex'])
    exp['Embarked'] = le_embarked.fit_transform(exp['Embarked'])

    num_cols = ['Age', 'Fare', 'FamilySize', 'FarePerPerson']

    scaler = MinMaxScaler()
    exp[num_cols] = scaler.fit_transform(exp[num_cols])

    return exp


def make_exp3(df):
    exp = df.copy()

    exp['Age'] = SimpleImputer(strategy='most_frequent').fit_transform(exp[['Age']])
    exp['Embarked'] = SimpleImputer(strategy='most_frequent').fit_transform(exp[['Embarked']]).ravel()

    exp = add_features(exp)

    exp = pd.get_dummies(
        exp,
        columns=['Sex', 'Embarked'],
        drop_first=True
    )

    num_cols = ['Age', 'Fare', 'FamilySize', 'FarePerPerson']

    scaler = RobustScaler()
    exp[num_cols] = scaler.fit_transform(exp[num_cols])

    return exp


def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    return {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }

experiments = {
    'Exp-1 Mean + OneHot + Standard': make_exp1(df),
    'Exp-2 Median + Label + MinMax': make_exp2(df),
    'Exp-3 MostFrequent + OneHot + Robust': make_exp3(df)
}

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

results = []

for exp_name, exp_data in experiments.items():

    X = exp_data.drop('Survived', axis=1)
    y = exp_data['Survived']

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    for model_name, model in models.items():
        score = evaluate_model(
            model,
            X_train,
            X_test,
            y_train,
            y_test
        )

        results.append({
            'Experiment': exp_name,
            'Model': model_name,
            **score
        })

result_df = pd.DataFrame(results)

print(result_df)


In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("train.csv")

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
target = "Survived"

base_df = df[features + [target]].copy()

base_df = base_df.dropna()

base_df["Sex"] = base_df["Sex"].map({"male": 0, "female": 1})
base_df["Embarked"] = base_df["Embarked"].map({"S": 0, "C": 1, "Q": 2})

X = base_df[features]
y = base_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42)
}

base_results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    base_results.append({
        "Experiment": "Base",
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

base_results_df = pd.DataFrame(base_results)
base_results_df

###Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

###GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 10],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

###SHAP

In [ ]:
import shap

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(
    shap_values,
    X_test
)